# contextkit — fit a prompt to budget without dropping the wrong thing

Naive truncation lops off the end of your prompt, which is usually the pinned instructions or the user's actual question. `Context` assembles blocks by priority, shrinks what it may, drops what it must, and hands back a receipt.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## 1 · Four blocks that do not fit

A pinned system prompt, a large retrieved-docs blob, 40 turns of history, and the user's question — together well over the 8,000-token budget.

In [ ]:
import main as recipe
from cendor.core import tokens

print(f"docs   : {tokens.count(recipe.DOCS, 'gpt-4o'):,} tokens")
print(f"history: {tokens.count(recipe.HISTORY, 'gpt-4o'):,} tokens")

## 2 · Assemble

`pin=True` means never dropped. `evict="truncate"` means shrink me. `evict="drop_oldest"` means peel my oldest entries. Priority decides who suffers first.

In [ ]:
ctx = recipe.build()
report = ctx.report()
print(report)

## 3 · The receipt is the point

Every block says what happened to it, in tokens.

In [ ]:
for d in report.decisions:
    print(f"  {d.role:<10} {d.action:<10} {d.tokens_before:>7,} -> {d.tokens_after:>7,}")
print(f"\nused {report.used:,} / {report.budget:,} (−{report.reserved_output:,} reserved)")

## 4 · Determinism

Identical inputs must give byte-identical output — otherwise a recorded cassette can never match on replay.

In [ ]:
identical = recipe.build().assemble() == recipe.build().assemble()
identical

## 5 · Prove it

In [ ]:
ok = report.used <= (report.budget - report.reserved_output)
assert ok, "assembled prompt must fit the budget"
assert identical, "assembly must be deterministic"
print("OK")